# 4. Full MACE Architecture

In the previous notebook, we proved that a single MACE interaction block is rotationally equivariant. 

Now, we will assemble the full `MACE` architecture. This involves:
1. **Embedding**: Converting atomic numbers ($Z$) into initial scalar node features.
2. **Message Passing**: Stacking multiple `SimpleMACEBlock`s to build up high-order equivariant correlations.
3. **Readout**: Filtering out the equivariant tensors (keeping only the invariant $L=0$ scalars), passing them through an MLP to predict atomic site energies, and summing them for total energy.
4. **Autograd**: Using PyTorch's `autograd.grad` to take the derivative of the total energy with respect to the atomic positions to predict **Forces**.

In [1]:
import os
import sys
import torch
import ase
from ase.build import molecule
from torch_geometric.data import Batch

sys.path.append(os.path.abspath('..'))
from src.data import atoms_to_pyg_data
from src.model import MACE

### Creating a Batch of Molecules

Let's create two different molecules (Benzene and Water) and batch them together using PyTorch Geometric.

In [2]:
atoms1 = molecule('C6H6')
atoms2 = molecule('H2O')

data1 = atoms_to_pyg_data(atoms1, cutoff=4.0)
data2 = atoms_to_pyg_data(atoms2, cutoff=4.0)

# Batch them together. PyG will automatically handle the disconnected graph components!
batch = Batch.from_data_list([data1, data2])

print(f"Batched Graph:")
print(batch)
print(f"\nBatch assignment vector: {batch.batch}")

Batched Graph:
DataBatch(edge_index=[2, 120], pos=[15, 3], z=[15], edge_vec=[120, 3], edge_len=[120], batch=[15], ptr=[3])

Batch assignment vector: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1])


### Initializing the Model

We initialize the full MACE architecture. Notice how we specify `l_max` (maximum spherical harmonic degree) and `num_blocks` (number of message passing layers).

In [3]:
model = MACE(
    num_elements=10,  # Max atomic number to expect (C=6, H=1, O=8)
    r_max=4.0,        # Must match the graph cutoff
    num_radial=8,
    l_max=2,
    num_blocks=2,     # Two interaction blocks
    node_dim=16       # Dimensionality of the internal irreps
)

print(model)
print(f"\nTotal trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

MACE(
  (node_embedding): Embedding(10, 16)
  (initial_projection): Linear(16x0e -> 16x0e+16x1o+16x2e | 256 weights)
  (radial_basis): BesselBasis(
    (envelope): PolynomialEnvelope()
  )
  (sh_basis): SphericalHarmonicsBasis()
  (blocks): ModuleList(
    (0-1): 2 x SimpleMACEBlock(
      (mp): EquivariantMessagePassing(
        (tp): FullyConnectedTensorProduct(16x0e+16x1o+16x2e x 1x0e+1x1o+1x2e -> 16x0e+16x1o+16x2e | 2816 paths | 2816 weights)
        (radial_net): Sequential(
          (0): Linear(in_features=8, out_features=64, bias=True)
          (1): SiLU()
          (2): Linear(in_features=64, out_features=2816, bias=True)
        )
      )
      (contraction): FullyConnectedTensorProduct(16x0e+16x1o+16x2e x 16x0e+16x1o+16x2e -> 16x0e+16x1o+16x2e | 45056 paths | 45056 weights)
      (update): Linear(16x0e+16x1o+16x2e -> 16x0e+16x1o+16x2e | 768 weights)
    )
  )
  (readout_linear): Linear(16x0e+16x1o+16x2e -> 16x0e | 256 weights)
  (readout_mlp): Sequential(
    (0): Linear(in

### Forward Pass: Predicting Energy & Forces

We pass the batched graph through the model. PyTorch Autograd does the heavy lifting of tracking the operations so we can compute the analytical force vectors.

In [4]:
# The forward pass computes energy and explicitly derives forces via autograd
predictions = model(batch)

energy = predictions["energy"]
forces = predictions["forces"]

print("Predicted Total Energy:")
print(energy) # Shape: [2, 1] (one scalar per molecule in the batch)

print("\nPredicted Forces (first 5 atoms):")
print(forces[:5]) # Shape: [total_atoms, 3]

print(f"\nForces shape exactly matches positions shape? {forces.shape == batch.pos.shape}")

Predicted Total Energy:
tensor([[-3.8736],
        [-0.3820]], grad_fn=<ScatterAddBackward0>)

Predicted Forces (first 5 atoms):
tensor([[-1.0477e-09,  1.7489e-01, -0.0000e+00],
        [ 1.5146e-01,  8.7443e-02, -0.0000e+00],
        [ 1.5146e-01, -8.7443e-02, -0.0000e+00],
        [-8.0327e-09, -1.7489e-01, -0.0000e+00],
        [-1.5146e-01, -8.7443e-02, -0.0000e+00]], grad_fn=<SliceBackward0>)

Forces shape exactly matches positions shape? True


Because forces are derived explicitly as $\mathbf{F}_i = -\nabla_{\mathbf{r}_i} E$, they are guaranteed to be mathematically consistent with the energy surface. Furthermore, because $E$ is an invariant scalar (proved in Notebook 3), the gradient $\nabla_{\mathbf{r}_i} E$ is mathematically guaranteed to be a valid equivariant vector field!

Next, we will look at how to train these weights using a dataset.